In [13]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, GridSearchCV, ShuffleSplit
from sklearn.metrics import accuracy_score
from sklearn.base import clone
from scipy.stats import mode

In [7]:
X_moons, y_moons = make_moons(n_samples=10000, noise=0.4)
X_train, X_test, y_train, y_test = train_test_split(X_moons, y_moons, test_size=0.2)

In [8]:
classifier = DecisionTreeClassifier()

In [9]:
parameters = {
    "max_leaf_nodes": [5, 10, 20, 50, 100, 200],
    "max_depth": [2, 3, 4, 5, 6, 7],
}

grid_search = GridSearchCV(classifier, parameters, cv=5, n_jobs=-1, verbose=1)

In [10]:
grid_search.fit(X_train, y_train)
print('Best Hyperparameters:', grid_search.best_params_)
print(f'Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}')

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print(f'Final Test Set Accuracy: {accuracy_score(y_test, y_pred):.4f}')

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best Hyperparameters: {'max_depth': 6, 'max_leaf_nodes': 20}
Best Cross-Validation Accuracy: 0.8564
Final Test Set Accuracy: 0.8640


In [11]:
shuffle_split_data = ShuffleSplit(n_splits=1000, test_size=None, train_size=100)

In [15]:
base_tree = clone(best_model)
forest = [clone(base_tree) for _ in range(1000)]
accuracies = []
predictions = []

# train each shuffled data part on a decision tree
for i, (train_idx, _) in enumerate(shuffle_split_data.split(X_train)):
    X_train_subset, y_train_subset = X_train[train_idx], y_train[train_idx]
    forest[i].fit(X_train_subset, y_train_subset)
    forest_pred = forest[i].predict(X_test)
    predictions.append(forest_pred)
    accuracy = accuracy_score(y_test, forest_pred)
    accuracies.append(accuracy)

print(f'Average Accuracy: {np.mean(accuracies)}')


Average Accuracy: 0.8016335


In [22]:
# get the more common prediction of the 1000 trees for each sample in the test set
# predicitons is (1000 rows, 2000 columns) where each row is the prediction of a single tree and each column is the prediction of a single sample from test set.
# so [0, 0] is the prediction of the first tree for the first sample in the test set
# axis 0 tells mode function to look vertically, eg. if 700 trees predicted 0 and 300 trees predicted 1 for first sample(from test set), then the first value of most_common will be 0
most_common, _ = mode(np.array(predictions), axis=0, keepdims=False)
print(most_common) 

[1 0 0 ... 1 1 0]


In [23]:
final_accuracy = accuracy_score(y_test, most_common)
print(f'Final Accuracy: {final_accuracy:.4f}')

Final Accuracy: 0.8680
